In [2]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np

# 1. Falls 'archive.zip' existiert, automatisch entpacken
if os.path.exists('archive.zip'):
    print("📦 Entpacke 'archive.zip'...")
    with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✅ Datei erfolgreich entpackt!\n")

# 2. Entpackte Datensatz-Datei suchen
all_files = glob.glob('*.csv') + glob.glob('*.xlsx')
data_files = [f for f in all_files if not f.startswith('cleaned_')]

if not data_files:
    print("❌ FEHLER: Keine CSV- oder Excel-Datei gefunden.")
else:
    filename = data_files[0]
    print(f"📄 Lade Datei: '{filename}'...")

    # 3. Einlesen je nach Dateiendung
    if filename.endswith('.csv'):
        df = pd.read_csv(filename, encoding='ISO-8859-1')
    else:
        df = pd.read_excel(filename)

    print("\n--- Ursprünglicher Datensatz ---")
    print(f"Zeilen: {df.shape[0]}, Spalten: {df.shape[1]}")

    # 4. Spaltennamen säubern und vereinheitlichen
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Customer ID': 'CustomerID',
        'Customer_ID': 'CustomerID',
        'Price': 'UnitPrice',
        'Invoice': 'InvoiceNo'
    })

    # 5. Zeilen ohne Kunden-ID entfernen
    df_clean = df.dropna(subset=['CustomerID']).copy()

    # 6. Datentypen korrigieren
    df_clean['CustomerID'] = df_clean['CustomerID'].astype(float).astype(int).astype(str)
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

    # 7. Stornierungen (Invoice beginnt mit 'C') & ungültige Mengen/Preise filtern
    df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
    df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

    # 8. Gesamtumsatz pro Position berechnen
    df_clean['TotalSum'] = df_clean['Quantity'] * df_clean['UnitPrice']

    print("\n--- Bereinigter Datensatz ---")
    print(f"Verbleibende Zeilen: {df_clean.shape[0]}")
    print(f"Eindeutige Kunden: {df_clean['CustomerID'].nunique()}")

    # 9. Für SQL & Tableau exportieren
    df_clean.to_csv('cleaned_online_retail_II.csv', index=False)
    print("\n🎉 ERFOLG: Datei 'cleaned_online_retail_II.csv' wurde im Ordner erstellt!")

📦 Entpacke 'archive.zip'...
✅ Datei erfolgreich entpackt!

📄 Lade Datei: 'online_retail_II.csv'...

--- Ursprünglicher Datensatz ---
Zeilen: 1067371, Spalten: 8

--- Bereinigter Datensatz ---
Verbleibende Zeilen: 805549
Eindeutige Kunden: 5878

🎉 ERFOLG: Datei 'cleaned_online_retail_II.csv' wurde im Ordner erstellt!


In [3]:
import sqlite3
import pandas as pd

# 1. Verbindung zu einer In-Memory-SQL-Datenbank herstellen
conn = sqlite3.connect('ecommerce.db')

# 2. Die eben erzeugte CSV in die SQL-Tabelle 'orders' laden
df_clean = pd.read_csv('cleaned_online_retail_II.csv')
df_clean.to_sql('orders', conn, if_exists='replace', index=False)

# 3. Das SQL-Skript zur RFM-Berechnung ausführen
sql_query = """
WITH max_date AS (
    -- Referenzdatum: Der aktuellste Tag im Datensatz
    SELECT MAX(DATE(InvoiceDate)) AS max_order_date FROM orders
),
rfm_base AS (
    -- Schritt A: Recency, Frequency und Monetary pro Kunde berechnen
    SELECT
        CustomerID,
        Country,
        MAX(DATE(InvoiceDate)) AS last_order_date,
        CAST(JULIANDAY((SELECT max_order_date FROM max_date)) - JULIANDAY(MAX(DATE(InvoiceDate))) AS INT) AS Recency,
        COUNT(DISTINCT InvoiceNo) AS Frequency,
        ROUND(SUM(TotalSum), 2) AS Monetary
    FROM orders
    GROUP BY CustomerID
),
rfm_scores AS (
    -- Schritt B: Per Window Function (NTILE) Scores von 1 bis 5 vergeben
    SELECT
        CustomerID,
        Country,
        Recency,
        Frequency,
        Monetary,
        NTILE(5) OVER (ORDER BY Recency DESC) AS R_Score,
        NTILE(5) OVER (ORDER BY Frequency ASC) AS F_Score,
        NTILE(5) OVER (ORDER BY Monetary ASC) AS M_Score
    FROM rfm_base
)
-- Schritt C: Kunden in verständliche Business-Segmente einteilen
SELECT
    CustomerID,
    Country,
    Recency,
    Frequency,
    Monetary,
    R_Score,
    F_Score,
    M_Score,
    CASE
        WHEN R_Score >= 4 AND F_Score >= 4 THEN 'Champions'
        WHEN R_Score >= 3 AND F_Score >= 3 THEN 'Loyal Customers'
        WHEN R_Score >= 3 AND F_Score < 3 THEN 'Promising / New'
        WHEN R_Score <= 2 AND F_Score >= 3 THEN 'At Risk / Retention Needed'
        ELSE 'Lost / Inactive'
    END AS Customer_Segment
FROM rfm_scores;
"""

# SQL-Ergebnis in DataFrame speichern & exportieren
df_rfm = pd.read_sql_query(sql_query, conn)
df_rfm.to_csv('rfm_customer_segments.csv', index=False)

print("🎉 SQL-ANALYSE ERFOLGREICH!")
print(f"Anzahl segmentierter Kunden: {len(df_rfm)}\n")
print("--- Vorschau der Kunden-Segmente ---")
print(df_rfm.head(10))
print("\n--- Verteilung der Segmente ---")
print(df_rfm['Customer_Segment'].value_counts())

🎉 SQL-ANALYSE ERFOLGREICH!
Anzahl segmentierter Kunden: 5878

--- Vorschau der Kunden-Segmente ---
   CustomerID         Country  Recency  Frequency  Monetary  R_Score  F_Score  \
0       17056  United Kingdom      738          1    128.60        1        1   
1       12636             USA      738          1    141.00        1        1   
2       17592  United Kingdom      738          1    148.30        1        1   
3       14654  United Kingdom      738          1    246.86        1        1   
4       13526  United Kingdom      738          2   1182.00        1        3   
5       15833  United Kingdom      737          1     80.40        1        1   
6       17606  United Kingdom      737          1     87.30        1        1   
7       17818  United Kingdom      737          1    130.18        1        1   
8       17909  United Kingdom      737          1    132.55        1        1   
9       14980  United Kingdom      737          1    150.68        1        1   

   M_Scor